<a href="https://colab.research.google.com/github/shanshiny06/econ3916-portfolio/blob/main/Problem%20Set%203/Econ_3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Econ 3916 PS3: The Number Nobody Can Reproduce
## Phase 1: Reproduce Both Numbers

In [14]:
!pip install duckdb --quiet

In [15]:
import numpy as np
import pandas as pd
import duckdb

rng = np.random.default_rng(3916)     # fixed seed so a grader can rerun and get the same numbers
N_CUSTOMERS = 3500

# Exactly 2,200 West customers (at least 2,000 required), shuffled among the other regions
region = np.array(["West"] * 2200 + ["East"] * 450 + ["North"] * 450 + ["South"] * 400)
rng.shuffle(region)

# Order counts: 1 + NegativeBinomial(n=0.8, p=0.12), mean near 7 with a long right tail
n_orders = 1 + rng.negative_binomial(n=0.8, p=0.12, size=N_CUSTOMERS)

# Revenue is driven by order count, which is what makes the fan-out move the average
revenue = np.maximum(60 * n_orders + rng.normal(400, 150, size=N_CUSTOMERS), 50)

signup_date = pd.Timestamp("2022-01-01") + pd.to_timedelta(rng.integers(0, 3 * 365, size=N_CUSTOMERS), unit="D")
customers_df = pd.DataFrame({"customer_id": np.arange(1, N_CUSTOMERS + 1), "region": region,
                             "revenue": revenue.round(2), "signup_date": signup_date})

# One row in orders for each of a customer's n_orders
order_customer = np.repeat(customers_df["customer_id"].to_numpy(), n_orders)
orders_df = pd.DataFrame({"order_id": np.arange(1, len(order_customer) + 1),
                          "customer_id": order_customer,
                          "order_date": pd.Timestamp("2025-01-01")
                                        + pd.to_timedelta(rng.integers(0, 365, size=len(order_customer)), unit="D")})

# Register the DataFrames with DuckDB and copy them into real tables
con = duckdb.connect()
con.register("customers_df", customers_df)
con.register("orders_df", orders_df)
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_df")

# Confirm both tables can be queried
con.execute("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM customers
    UNION ALL
    SELECT 'orders', COUNT(*) FROM orders
""").df()

,table_name,n_rows
0,customers,3500
1,orders,23111


In [16]:
# The low number: one row per customer, averaged over the customers table
query_customers = """
    SELECT COUNT(*)     AS rows_averaged,
           AVG(revenue) AS avg_revenue
    FROM customers
    WHERE region = 'West'
"""

# The high number: the same column after joining orders, so each customer appears once per order
query_joined = """
    SELECT COUNT(*)       AS rows_averaged,
           AVG(c.revenue) AS avg_revenue
    FROM customers AS c
    JOIN orders    AS o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

low = con.execute(query_customers).df()
high = con.execute(query_joined).df()
comparison = pd.concat([low.assign(query="customers only"), high.assign(query="customers JOIN orders")])
print(comparison[["query", "rows_averaged", "avg_revenue"]].round(2).to_string(index=False))
print(f"\nRatio, joined / un-joined: {high['avg_revenue'][0] / low['avg_revenue'][0]:.3f}")

                query  rows_averaged  avg_revenue
       customers only           2200       800.82
customers JOIN orders          14655      1214.32

Ratio, joined / un-joined: 1.516


The correct number is the un-joined one, about 801 dollars. The question is "average revenue per customer", so the unit of observation is the customer, and each customer must count exactly once. revenue is a customer-level column: it lives on the customers table, one value per customer. The un-joined query averages over 2,200 rows, one per West customer, which is exactly that unit.

The joined query averages the same column over about 14,655 rows, one per order. Joining orders repeats each customer's revenue once for every order they placed, so the result is a weighted average in which each customer's weight is their order count. Because revenue rises with order count, heavy buyers are counted many times and the average is pushed up to about 1,214 dollars. That number answers a different question, roughly "the average revenue of the customer behind a typical order", not "the average revenue per customer". Neither query has an arithmetic error; one of them silently changed the unit of observation from customers to orders.

In [17]:
def check_join(con, base_sql, joined_sql, key="customer_id"):
    """Warn if a join changes the row count, or repeats the key, relative to the query it started from."""
    n_base = con.execute(f"SELECT COUNT(*) FROM ({base_sql})").fetchone()[0]
    n_joined, n_keys = con.execute(f"SELECT COUNT(*), COUNT(DISTINCT {key}) FROM ({joined_sql})").fetchone()
    if n_joined != n_base or n_keys != n_joined:
        print(f"WARNING: the join changed the row count from {n_base:,} to {n_joined:,}, "
              f"and {key} now appears {n_joined / n_keys:.2f} times on average. "
              "Any average over a column from the left table is now weighted by the join.")
        return False
    print(f"OK: {n_joined:,} rows before and after the join; each {key} appears once.")
    return True

# The rows the dashboard averaged over, before any aggregation
west_customers = "SELECT customer_id, revenue FROM customers WHERE region = 'West'"

# The bad join: one row per order
bad_join = """
    SELECT c.customer_id, c.revenue
    FROM customers AS c
    JOIN orders    AS o ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

# A good join: aggregate orders to one row per customer first, then join
good_join = """
    SELECT c.customer_id, c.revenue, o.n_orders
    FROM customers AS c
    JOIN (SELECT customer_id, COUNT(*) AS n_orders FROM orders GROUP BY customer_id) AS o
      ON c.customer_id = o.customer_id
    WHERE c.region = 'West'
"""

print("Bad join:")
check_join(con, west_customers, bad_join)
print("\nGood join:")
check_join(con, west_customers, good_join)

Bad join:

Good join:
OK: 2,200 rows before and after the join; each customer_id appears once.


True

The check compares the rows a query starts from with the rows it ends up with after a join, and also checks that the key is still unique. On the dashboard's query it fires: 2,200 West customers become about 14,655 rows, with each customer repeated about 6.7 times. On the corrected query, which first aggregates orders to one row per customer and then joins, it stays silent: 2,200 rows in and 2,200 rows out. The rule it encodes is simple: before averaging a column, confirm that the join did not change the number of rows at the level that column was measured.

## Phase 2: The Rows That Disappear

In [18]:
# A separate seeded generator, so this step gives the same rows however often earlier cells are rerun
rng_null = np.random.default_rng(2)
null_ids = rng_null.choice(customers_df["customer_id"].to_numpy(),
                           size=round(0.15 * N_CUSTOMERS), replace=False)
con.register("null_ids_df", pd.DataFrame({"customer_id": null_ids}))

# Customers onboarded before region was a mandatory field
con.execute("UPDATE customers SET region = NULL WHERE customer_id IN (SELECT customer_id FROM null_ids_df)")

con.execute("""
    SELECT region, COUNT(*) AS n_customers
    FROM customers
    GROUP BY region
    ORDER BY region NULLS LAST
""").df()

,region,n_customers
0,East,369
1,North,375
2,South,342
3,West,1889
4,None,525


In [19]:
# What the analyst wrote: "everyone outside the West"
query_naive = "SELECT COUNT(*) AS n_customers FROM customers WHERE region != 'West'"

# What the analyst meant: every customer not known to be in the West, including unknown regions
query_meant = "SELECT COUNT(*) AS n_customers FROM customers WHERE region IS DISTINCT FROM 'West'"

n_naive = con.execute(query_naive).fetchone()[0]
n_meant = con.execute(query_meant).fetchone()[0]
n_null = con.execute("SELECT COUNT(*) FROM customers WHERE region IS NULL").fetchone()[0]

print(f"WHERE region != 'West':                 {n_naive:,} customers")
print(f"WHERE region IS DISTINCT FROM 'West':   {n_meant:,} customers")
print(f"Difference:                             {n_meant - n_naive:,}   (customers with a NULL region: {n_null:,})")

WHERE region != 'West':                 1,086 customers
WHERE region IS DISTINCT FROM 'West':   1,611 customers
Difference:                             525   (customers with a NULL region: 525)


WHERE region != 'West' returns 1,086 customers; the query the analyst meant, WHERE region IS DISTINCT FROM 'West', returns 1,611. The difference is 525, exactly the customers whose region is NULL.

The mechanism: in SQL, comparing NULL with anything gives NULL (unknown), not TRUE or FALSE, and WHERE keeps only rows where the condition is TRUE, so every customer with an unknown region silently fails both region = 'West' and region != 'West'.

The consequence is that every regional comparison in the company is computed on a population that quietly excludes 15% of customers, and no query ever reports them as missing. If those customers are systematically different, for example newer, smaller or more likely to churn, then every regional average, total and growth rate describes only the customers whose region happened to be recorded, not the regions themselves. The regions' totals no longer add up to the company total, and a comparison such as "West versus everyone else" compares West with only the known part of everyone else. Worse, the bias does not have to be the same in every region: if the unknowns are concentrated in one region's history, that region looks better or worse than it is, and no one can see why. This is the same selection problem as Problem Set 2: the dataset being analysed is only the rows that passed a filter, and the filter is related to the outcome.

In [20]:
query_window = """
    SELECT customer_id,
           region,
           revenue,
           AVG(revenue) OVER (PARTITION BY region)           AS region_avg_revenue,
           revenue - AVG(revenue) OVER (PARTITION BY region) AS above_region_avg
    FROM customers
    ORDER BY customer_id
"""
window_df = con.execute(query_window).df()

n_customers = con.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
print(f"Rows in customers: {n_customers:,}   Rows returned: {len(window_df):,}")
window_df.head(10).round(2)

Rows in customers: 3,500   Rows returned: 3,500


,customer_id,region,revenue,region_avg_revenue,above_region_avg
0,1,South,354.91,792.22,-437.31
1,2,West,478.04,804.97,-326.93
2,3,West,469.30,804.97,-335.67
3,4,East,782.39,795.34,-12.95
4,5,North,985.15,794.85,190.30
5,6,East,1141.91,795.34,346.57
6,7,West,342.82,804.97,-462.15
7,8,None,723.24,778.87,-55.63
8,9,West,149.98,804.97,-654.99
9,10,West,720.95,804.97,-84.02


The window function computes each region's average revenue and attaches it to every customer's row, so revenue and the regional benchmark sit side by side. Unlike GROUP BY, which collapses each region to a single row, PARTITION BY keeps every row: the query returns 3,500 rows, the same as the customers table, with no join and no subquery. One detail matters here: the 525 customers with a NULL region are placed in a partition of their own, so they are compared with the average of the other unknown-region customers rather than silently dropped.

## Phase 3: The Causal Question

In [21]:
import statsmodels.api as sm
from scipy.special import expit

rng_causal = np.random.default_rng(3916)   # its own generator, separate from the warehouse
n = 5000
TRUE_EFFECT = 50                           # the programme's true effect on revenue, in dollars

# The confounder: existing account size (standardised). Bigger accounts were more likely
# to be in the West, and bigger accounts also bring in more revenue on their own
account_size = rng_causal.normal(0, 1, n)
programme = (rng_causal.uniform(size=n) < expit(-0.2 + 1.5 * account_size)).astype(int)   # 1 = West
revenue_sim = (800 + TRUE_EFFECT * programme + 250 * account_size
               + rng_causal.normal(0, 150, n))

sim = pd.DataFrame({"programme": programme, "account_size": account_size, "revenue": revenue_sim})

def programme_effect(controls):
    """Return the OLS coefficient on programme, controlling for the listed columns of sim."""
    X = sm.add_constant(sim[["programme"] + controls])
    return sm.OLS(sim["revenue"], X).fit().params["programme"]

naive = programme_effect([])
adjusted = programme_effect(["account_size"])

print(f"Share of customers in the programme: {programme.mean():.1%}")
print(f"{'estimate':<40}{'dollars':>10}")
print("-" * 50)
print(f"{'True effect':<40}{TRUE_EFFECT:>10.1f}")
print(f"{'Naive: revenue ~ programme':<40}{naive:>10.1f}")
print(f"{'Adjusted: + account_size':<40}{adjusted:>10.1f}")

Share of customers in the programme: 46.5%
estimate                                   dollars
--------------------------------------------------
True effect                                   50.0
Naive: revenue ~ programme                   334.8
Adjusted: + account_size                      56.6


In this simulated world the programme's true effect is 50 dollars per customer. The naive comparison, regressing revenue on programme alone, says about 335 dollars, almost seven times the truth. The gap is not the programme: larger existing accounts were more likely to be placed in the West, and larger accounts bring in more revenue whether or not they get account management. The naive comparison credits the programme with the difference in account size. Once account size is held constant, the estimate falls to about 57 dollars, close to the true 50; the remaining difference is sampling noise. The adjustment works here because I built the world and know that account size is the only thing driving both assignment and revenue. In the real data, the case for the programme rests on the claim that nothing else does.

In [22]:
# Support tickets are caused by BOTH the programme (account managers encourage customers
# to log issues) and by revenue (bigger spenders raise more tickets)
sim["support_tickets"] = (2 + 1.0 * sim["programme"] + 0.01 * (sim["revenue"] - 800)
                          + rng_causal.normal(0, 1, n))

with_tickets = programme_effect(["account_size", "support_tickets"])

print(f"{'estimate':<40}{'dollars':>10}")
print("-" * 50)
print(f"{'True effect':<40}{TRUE_EFFECT:>10.1f}")
print(f"{'Naive: revenue ~ programme':<40}{naive:>10.1f}")
print(f"{'Adjusted: + account_size':<40}{adjusted:>10.1f}")
print(f"{'Adjusted: + account_size + tickets':<40}{with_tickets:>10.1f}")

estimate                                   dollars
--------------------------------------------------
True effect                                   50.0
Naive: revenue ~ programme                   334.8
Adjusted: + account_size                      56.6
Adjusted: + account_size + tickets           -54.7


Adding support tickets as a control moves the estimate from about 57 dollars, close to the true 50, to about -55: the sign flips, and a programme that raises revenue now appears to lower it. Support tickets are not a background characteristic of the customer; in this simulation they are an outcome of both the programme and revenue, so holding them fixed compares the wrong customers.

**Paragraph for the director**

Controlling for support tickets sounds sensible, but it made the answer worse, because tickets are not a fixed trait of a customer: they go up both when a customer joins the programme (account managers encourage customers to raise issues) and when a customer spends more. So when we compare programme and non-programme customers "with the same number of tickets", we are not comparing like with like. A non-programme customer who files as many tickets as a programme customer must be filing them for some other reason, and the main other reason is that they spend more. Among customers with equal tickets, the ones outside the programme are therefore the bigger spenders, and the programme looks as if it lowers revenue, even though it raises it by about 50 dollars per customer. The general rule is that we should only control for things that were already true before the programme started, like account size, and never for things the programme itself changes.

## Phase 4: AI Expansion — The Co-Pilot Era
### Task 4.1: The query reviewer

In [23]:
"""
review_query(con, sql) -> list[ReviewWarning]

Data-aware review of a single SELECT against a live DuckDB connection.
Dependencies: duckdb, pandas (plus the standard library).

Checks
  JOIN_ROW_COUNT_CHANGE      a join makes the result grain differ from the driving table
  LEFT_JOIN_NULLIFIED        WHERE filters the nullable side of an outer join (silently becomes INNER)
  NULL_FILTER_DROPS_UNKNOWNS WHERE predicate on a nullable column discards NULL rows
  AGGREGATE_OVER_FANOUT      SUM/AVG/COUNT over rows that a join has duplicated
Meta (so an empty list always means "checked and clean", never "gave up quietly")
  PARSE_ERROR, UNSUPPORTED, CHECK_FAILED

Parsing uses DuckDB's own parser via json_serialize_sql(), so there is no
regex guessing about SQL and no extra dependency like sqlglot.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field

import duckdb
import pandas as pd


@dataclass
class ReviewWarning:
    code: str
    severity: str  # "error" | "warning" | "info"
    message: str
    details: dict = field(default_factory=dict)


def warnings_to_frame(warnings: list[ReviewWarning]) -> pd.DataFrame:
    """Convenience for notebooks / assertions on columns."""
    return pd.DataFrame([asdict(w) for w in warnings],
                        columns=["code", "severity", "message", "details"])


class _Unsupported(Exception):
    pass


# --------------------------------------------------------------------------
# AST helpers
# --------------------------------------------------------------------------

# Aggregates whose result changes when input rows are duplicated.
# MIN/MAX/ANY_VALUE are deliberately absent: duplicating a row can't move an extreme.
_FANOUT_SENSITIVE_AGGS = {
    "sum", "avg", "mean", "count", "count_star", "median", "quantile_cont",
    "string_agg", "list", "array_agg", "stddev", "stddev_samp", "variance",
}

# Predicates that treat NULL as a value rather than as "unknown".
_NULL_SAFE_COMPARES = {"COMPARE_DISTINCT_FROM", "COMPARE_NOT_DISTINCT_FROM"}
_NULL_AWARE_TYPES = {"OPERATOR_IS_NULL", "OPERATOR_IS_NOT_NULL", "OPERATOR_COALESCE"}
_NULL_AWARE_FUNCS = {"coalesce", "ifnull", "isnull", "nvl"}

_NEGATIVE_COMPARES = {"COMPARE_NOTEQUAL", "COMPARE_NOT_IN"}
_PATTERN_FUNCS = {"~~", "!~~", "~~*", "!~~*", "like", "ilike", "not_like",
                  "not_ilike", "regexp_matches", "prefix", "suffix", "contains"}
_NEGATIVE_FUNCS = {"!~~", "!~~*", "not_like", "not_ilike"}


def _q(ident: str) -> str:
    return '"' + str(ident).replace('"', '""') + '"'


def _children(node):
    for v in node.values():
        if isinstance(v, dict) and "class" in v:
            yield v
        elif isinstance(v, list):
            for x in v:
                if isinstance(x, dict) and "class" in x:
                    yield x


def _walk(node):
    # Subqueries hold a "subquery" statement (no "class" key), so they are not
    # descended into. That keeps column resolution in the outer query's scope.
    if not isinstance(node, dict):
        return
    yield node
    for c in _children(node):
        yield from _walk(c)


def _is_colref(n) -> bool:
    return isinstance(n, dict) and n.get("class") == "COLUMN_REF"


def _direct_columns(n):
    """Column refs that are immediate operands (looking through CAST, which preserves NULL)."""
    for c in _children(n):
        while c.get("class") == "CAST" and isinstance(c.get("child"), dict):
            c = c["child"]
        if _is_colref(c):
            yield c


def _parse_select(con, sql: str) -> dict:
    raw = con.execute("SELECT json_serialize_sql(?::VARCHAR)", [sql]).fetchone()[0]
    tree = json.loads(raw)
    if tree.get("error"):
        raise ValueError(tree.get("error_message", "parser error"))
    stmts = tree.get("statements", [])
    if len(stmts) != 1:
        raise _Unsupported(f"expected exactly one statement, got {len(stmts)}")
    node = stmts[0].get("node", {})
    if node.get("type") != "SELECT_NODE":
        raise _Unsupported(f"top-level node is {node.get('type')}; only plain SELECT "
                           "(no UNION/INTERSECT) is reviewed")
    return node


def _flatten_from(ref, tables: dict, joins: list, ctes: set) -> list[str]:
    """Collect alias -> table and the join tree. Returns aliases under this ref."""
    if ref is None:
        return []
    t = ref.get("type")
    if t == "BASE_TABLE":
        name = ref["table_name"]
        if name.lower() in ctes:
            raise _Unsupported(f"'{name}' is a CTE; cardinality checks need base tables")
        alias = (ref.get("alias") or name).lower()
        parts = [p for p in (ref.get("catalog_name"), ref.get("schema_name"), name) if p]
        tables[alias] = {"name": name, "fq": ".".join(_q(p) for p in parts)}
        return [alias]
    if t == "JOIN":
        left = _flatten_from(ref.get("left"), tables, joins, ctes)
        right = _flatten_from(ref.get("right"), tables, joins, ctes)
        joins.append({
            "join_type": (ref.get("join_type") or "INNER").upper(),
            "ref_type": (ref.get("ref_type") or "REGULAR").upper(),
            "condition": ref.get("condition"),
            "using": [u.lower() for u in ref.get("using_columns") or []],
            "left": left,
            "right": right,
        })
        return left + right
    raise _Unsupported(f"FROM item of type {t} (subquery/table function) is not reviewed")


def _describe(con, fq: str) -> dict:
    df = con.execute(f"DESCRIBE {fq}").df()
    return {
        str(r.column_name).lower(): {"type": r.column_type, "nullable": str(r.null).upper() == "YES"}
        for r in df.itertuples()
    }


def _resolve(colref, aliases, cols):
    names = [str(n).lower() for n in colref.get("column_names", [])]
    if not names:
        return None
    if len(names) >= 2:
        a, c = names[-2], names[-1]
        return (a, c) if a in aliases and c in cols.get(a, {}) else None
    owners = [a for a in aliases if names[0] in cols.get(a, {})]
    return (owners[0], names[0]) if len(owners) == 1 else None


def _equi_keys(join, cols):
    """[(left_alias, left_col, right_alias, right_col)], plus whether the ON clause was purely equalities."""
    left, right = set(join["left"]), set(join["right"])
    keys, pure = [], True
    if join["using"]:
        for c in join["using"]:
            la = [a for a in join["left"] if c in cols[a]]
            ra = [a for a in join["right"] if c in cols[a]]
            if la and ra:
                keys.append((la[0], c, ra[0], c))
        return keys, pure

    def visit(n):
        nonlocal pure
        if n is None:
            return
        if n.get("type") == "CONJUNCTION_AND":
            for c in n.get("children", []):
                visit(c)
            return
        if n.get("type") == "COMPARE_EQUAL" and _is_colref(n.get("left")) and _is_colref(n.get("right")):
            a = _resolve(n["left"], left | right, cols)
            b = _resolve(n["right"], left | right, cols)
            if a and b:
                if a[0] in left and b[0] in right:
                    keys.append((a[0], a[1], b[0], b[1]))
                    return
                if b[0] in left and a[0] in right:
                    keys.append((b[0], b[1], a[0], a[1]))
                    return
        pure = False  # non-equi or single-table predicate inside ON

    visit(join["condition"])
    return keys, pure


# --------------------------------------------------------------------------
# Check 1: join row-count change (fan-out or drop)
# --------------------------------------------------------------------------
#
# HOW FAN-OUT IS DETECTED WITHOUT RUNNING THE JOIN (let alone twice):
#
# For an equi-join, the output row count is fully determined by the key
# histograms of the two sides:
#
#     |L JOIN R|  =  SUM over key k of  count_L(k) * count_R(k)
#
# So instead of materialising L JOIN R (rows = up to |L| * max_mult) and then
# counting it, we GROUP BY the key on each side independently, giving one row
# per distinct key, and join those two small histograms. The cost scales with
# distinct keys, not with the joined row count, and the answer is exact, not
# sampled. From the same histogram join we also get:
#   - max multiplicity per side  -> does either side repeat the other's rows?
#   - matched row totals         -> how many rows an INNER join throws away
# NULL keys fall out naturally: they form their own GROUP BY bucket but can
# never satisfy `=` in the histogram join, exactly as in the real join.
#
# The same histogram also prices the damage for SUM: joining a measure column
# to count_R(k) and computing SUM(measure * count_R(k)) reproduces what the
# fanned-out SUM would report, again without building the fanned-out rows.

def _join_stats(con, join, tables, cols, warnings):
    jt = join["join_type"]
    keys, pure = _equi_keys(join, cols)
    label = f"{jt} JOIN {', '.join(join['right'])}"
    if not keys:
        warnings.append(ReviewWarning(
            "JOIN_ROW_COUNT_CHANGE", "warning",
            f"{label} has no column-equality keys (cross/theta join); output row count "
            "is not bounded by either input and was not verified.",
            {"join_type": jt, "right": join["right"]}))
        return None
    la, ra = {k[0] for k in keys}, {k[2] for k in keys}
    if len(la) != 1 or len(ra) != 1:
        warnings.append(ReviewWarning(
            "UNSUPPORTED", "info",
            f"{label}: join keys span several tables on one side; cardinality not checked.",
            {"keys": keys}))
        return None
    L, R = la.pop(), ra.pop()
    lk, rk = [k[1] for k in keys], [k[3] for k in keys]
    n = len(keys)
    lsel = ", ".join(f"{_q(c)} AS k{i}" for i, c in enumerate(lk))
    rsel = ", ".join(f"{_q(c)} AS k{i}" for i, c in enumerate(rk))
    on = " AND ".join(f"l.k{i} = r.k{i}" for i in range(n))
    nn = " AND ".join(f"k{i} IS NOT NULL" for i in range(n))
    q = f"""
        WITH l AS (SELECT {lsel}, count(*) AS n FROM {tables[L]['fq']} GROUP BY ALL),
             r AS (SELECT {rsel}, count(*) AS n FROM {tables[R]['fq']} GROUP BY ALL),
             m AS (SELECT l.n AS ln, r.n AS rn FROM l JOIN r ON {on})
        SELECT
          (SELECT coalesce(sum(n), 0) FROM l)                  AS left_rows,
          (SELECT coalesce(sum(n), 0) FROM r)                  AS right_rows,
          (SELECT coalesce(max(n), 0) FROM l WHERE {nn})       AS left_max_mult,
          (SELECT coalesce(max(n), 0) FROM r WHERE {nn})       AS right_max_mult,
          (SELECT count(*) FROM l WHERE {nn} AND n > 1)        AS left_dup_keys,
          (SELECT count(*) FROM r WHERE {nn} AND n > 1)        AS right_dup_keys,
          (SELECT coalesce(sum(ln * rn), 0) FROM m)            AS inner_rows,
          (SELECT coalesce(sum(ln), 0) FROM m)                 AS left_matched,
          (SELECT coalesce(sum(rn), 0) FROM m)                 AS right_matched
    """
    row = con.execute(q).df().iloc[0]
    s = {k: int(v) for k, v in row.items()}
    s.update(join_type=jt, L=L, R=R, lk=lk, rk=rk,
             left_aliases=join["left"], right_aliases=join["right"])
    unmatched_l = s["left_rows"] - s["left_matched"]
    unmatched_r = s["right_rows"] - s["right_matched"]
    expected = {
        "INNER": s["inner_rows"],
        "LEFT": s["inner_rows"] + unmatched_l,
        "RIGHT": s["inner_rows"] + unmatched_r,
        "OUTER": s["inner_rows"] + unmatched_l + unmatched_r,
        "FULL": s["inner_rows"] + unmatched_l + unmatched_r,
    }.get(jt, s["inner_rows"])
    baseline = s["right_rows"] if jt == "RIGHT" else s["left_rows"]
    s.update(unmatched_left=unmatched_l, unmatched_right=unmatched_r,
             expected_rows=expected, baseline_rows=baseline)

    # Compare components, not just totals: +1 duplicated row and -1 dropped row
    # nets to the same count but is two separate errors in every aggregate.
    dropped = unmatched_l if jt in ("INNER", "RIGHT") else 0
    grows = s["right_dup_keys"] > 0 or (s["left_dup_keys"] > 0 and jt in ("RIGHT", "FULL", "OUTER"))
    if expected != baseline or grows or dropped:
        reasons = []
        if s["right_dup_keys"]:
            reasons.append(f"{s['right_dup_keys']} key value(s) repeat in {tables[R]['name']} "
                           f"(up to {s['right_max_mult']} rows per key)")
        if s["left_dup_keys"] and jt in ("RIGHT", "FULL", "OUTER"):
            reasons.append(f"{s['left_dup_keys']} key value(s) repeat in {tables[L]['name']}")
        if jt in ("INNER", "RIGHT") and unmatched_l:
            reasons.append(f"{unmatched_l} {tables[L]['name']} row(s) have no match "
                           "(including NULL keys) and are dropped")
        warnings.append(ReviewWarning(
            "JOIN_ROW_COUNT_CHANGE", "warning",
            f"{label} on {', '.join(f'{L}.{a} = {R}.{b}' for a, b in zip(lk, rk))} changes row "
            f"count {baseline:,} -> {expected:,} ({expected - baseline:+,}): " + "; ".join(reasons)
            + ("" if pure else " [ON clause has extra non-equality terms; count is an upper bound]"),
            {k: v for k, v in s.items() if k not in ("left_aliases", "right_aliases")}))
    return s


# --------------------------------------------------------------------------
# Check 2: filters on nullable columns
# --------------------------------------------------------------------------
#
# WHY THIS IS A CORRECTNESS ISSUE, NOT STYLE:
#
# SQL predicates are three-valued: TRUE, FALSE, UNKNOWN. Any comparison with
# NULL (=, <>, <, IN, NOT IN, LIKE...) yields UNKNOWN, and WHERE keeps only
# TRUE rows. So `WHERE status <> 'cancelled'` does NOT mean "everything except
# cancelled"; it means "rows whose status is known and is not cancelled".
# Rows with an unknown status vanish, no error is raised, and the total just
# comes out lower. The query's meaning differs from its reading, so the
# number on the dashboard is wrong, not merely written in an unidiomatic way.
# Worse, the gap is data-dependent: the query is right today with zero NULLs
# and wrong after tomorrow's load introduces some.
#
# Severity follows intent. A positive predicate (status = 'paid') excluding
# NULLs is almost always what the author meant, so it is "info". A negative
# predicate (<>, NOT IN, NOT LIKE) is written to *keep* everything else, so
# losing NULLs contradicts its intent and is a "warning". A predicate on the
# nullable side of a LEFT JOIN is an "error": it turns the LEFT JOIN into an
# INNER JOIN, dropping every unmatched row regardless of the column's own data.
#
# Columns already wrapped in IS [NOT] NULL / COALESCE in the WHERE are treated
# as deliberately handled.

def _check_where(con, node, tables, cols, joins, stats, warnings):
    where = node.get("where_clause")
    if not where:
        return
    aliases = list(tables)
    handled, filtered = set(), {}

    def visit(n, negated):
        if not isinstance(n, dict):
            return
        t = n.get("type", "") or ""
        fn = (n.get("function_name") or "").lower()
        if t in _NULL_AWARE_TYPES or fn in _NULL_AWARE_FUNCS:
            for c in _walk(n):
                if _is_colref(c) and (r := _resolve(c, aliases, cols)):
                    handled.add(r)
            return
        if t == "OPERATOR_NOT":
            for c in _children(n):
                visit(c, not negated)
            return
        is_cmp = t.startswith("COMPARE_") and t not in _NULL_SAFE_COMPARES
        if is_cmp or fn in _PATTERN_FUNCS:
            neg = (t in _NEGATIVE_COMPARES or fn in _NEGATIVE_FUNCS) != negated
            for c in _direct_columns(n):
                if r := _resolve(c, aliases, cols):
                    filtered.setdefault(r, {"negative": False, "ops": set()})
                    filtered[r]["negative"] |= neg
                    filtered[r]["ops"].add(t or fn)
        for c in _children(n):
            visit(c, negated)

    visit(where, False)

    nullable_side = {}
    for j in joins:
        if j["join_type"] in ("LEFT", "FULL", "OUTER"):
            for a in j["right"]:
                nullable_side[a] = j
        if j["join_type"] in ("RIGHT", "FULL", "OUTER"):
            for a in j["left"]:
                nullable_side[a] = j

    for (alias, col), info in filtered.items():
        if (alias, col) in handled:
            continue
        tname, ops = tables[alias]["name"], sorted(info["ops"])
        if alias in nullable_side:
            j = nullable_side[alias]
            s = next((s for s in stats if s and alias in (s["R"], s["L"])), None)
            lost = s["unmatched_left"] if s and alias == s["R"] else (s["unmatched_right"] if s else None)
            warnings.append(ReviewWarning(
                "LEFT_JOIN_NULLIFIED", "error",
                f"WHERE filters {alias}.{col}, which is on the NULL-extended side of a "
                f"{j['join_type']} JOIN. Unmatched rows have {alias}.{col} = NULL, the predicate is "
                "UNKNOWN, and they are dropped: the outer join is effectively INNER"
                + (f" ({lost:,} row(s) lost)." if lost is not None else ".")
                + " Move the predicate into the ON clause or handle NULL explicitly.",
                {"alias": alias, "table": tname, "column": col, "ops": ops, "rows_lost": lost}))
            continue
        meta = cols[alias][col]
        nulls, total = con.execute(
            f"SELECT count(*) FILTER (WHERE {_q(col)} IS NULL), count(*) FROM {tables[alias]['fq']}"
        ).fetchone()
        nulls, total = int(nulls), int(total)
        if nulls == 0 and not meta["nullable"]:
            continue
        if nulls == 0:
            sev, msg = "info", (f"{alias}.{col} is declared nullable and has no NULLs today; "
                                f"the {', '.join(ops)} predicate will start dropping rows when it does.")
        else:
            sev = "warning" if info["negative"] else "info"
            msg = (f"WHERE {', '.join(ops)} on {alias}.{col} silently drops {nulls:,} of {total:,} "
                   f"{tname} row(s) where it is NULL (three-valued logic: NULL comparisons are UNKNOWN, "
                   "and WHERE keeps only TRUE)."
                   + (" The predicate is negative, so the author almost certainly meant to keep them."
                      if info["negative"] else ""))
        warnings.append(ReviewWarning("NULL_FILTER_DROPS_UNKNOWNS", sev, msg,
                                      {"alias": alias, "table": tname, "column": col, "ops": ops,
                                       "null_rows": nulls, "total_rows": total,
                                       "negative_predicate": info["negative"]}))


# --------------------------------------------------------------------------
# Check 3: aggregates over fanned-out rows
# --------------------------------------------------------------------------

def _check_aggregates(con, node, tables, cols, stats, warnings):
    # Which aliases have their rows repeated by some join?
    fanned = {}
    for s in stats:
        if not s:
            continue
        if s["right_max_mult"] > 1 and s["join_type"] in ("INNER", "LEFT", "FULL", "OUTER"):
            for a in s["left_aliases"]:
                fanned.setdefault(a, s)
        if s["left_max_mult"] > 1 and s["join_type"] in ("INNER", "RIGHT", "FULL", "OUTER"):
            for a in s["right_aliases"]:
                fanned.setdefault(a, s)
    if not fanned:
        return

    exprs = list(node.get("select_list") or [])
    if node.get("having"):
        exprs.append(node["having"])
    aliases = list(tables)
    for e in exprs:
        for f in _walk(e):
            fn = (f.get("function_name") or "").lower()
            if f.get("class") != "FUNCTION" or fn not in _FANOUT_SENSITIVE_AGGS or f.get("distinct"):
                continue
            if fn == "count_star":
                s = next(iter(fanned.values()))
                warnings.append(ReviewWarning(
                    "AGGREGATE_OVER_FANOUT", "error",
                    f"COUNT(*) counts joined rows, and {tables[s['R']]['name']} repeats "
                    f"{tables[s['L']]['name']} rows; it is not a count of {tables[s['L']]['name']}.",
                    {"function": fn, "join": f"{s['L']} -> {s['R']}"}))
                continue
            for c in _walk(f):
                r = _resolve(c, aliases, cols) if _is_colref(c) else None
                if not r or r[0] not in fanned:
                    continue
                s = fanned[r[0]]
                details = {"function": fn, "alias": r[0], "column": r[1],
                           "join": f"{s['L']} -> {s['R']}", "max_multiplicity":
                           max(s["right_max_mult"], s["left_max_mult"])}
                msg = (f"{fn.upper()}({r[0]}.{r[1]}) runs after a join that duplicates "
                       f"{tables[r[0]]['name']} rows (up to {details['max_multiplicity']}x)")
                arg = (f.get("children") or [None])[0]
                while isinstance(arg, dict) and arg.get("class") == "CAST":
                    arg = arg.get("child")
                if fn == "sum" and _is_colref(arg) and r[0] == s["L"] and s["right_max_mult"] > 1:
                    d = _price_sum_inflation(con, tables, s, r[1])
                    details.update(d)
                    msg += (f"; the joined SUM overstates by {d['overstatement']:,} "
                            f"({d['true_total']:,} -> {d['joined_total']:,}, before WHERE filters)")
                warnings.append(ReviewWarning("AGGREGATE_OVER_FANOUT", "error", msg + ".", details))


def _price_sum_inflation(con, tables, s, col):
    # Same histogram trick: multiply each measure by how many partner rows it
    # will be paired with, instead of materialising the pairs.
    n = len(s["lk"])
    rsel = ", ".join(f"{_q(c)} AS k{i}" for i, c in enumerate(s["rk"]))
    on = " AND ".join(f"_b.{_q(c)} = r.k{i}" for i, c in enumerate(s["lk"]))
    m = f"_b.{_q(col)}"
    q = f"""
        WITH r AS (SELECT {rsel}, count(*) AS n FROM {tables[s['R']]['fq']} GROUP BY ALL)
        SELECT sum({m})                                         AS base_total,
               sum(CASE WHEN r.n IS NOT NULL THEN {m} END)      AS matched_total,
               sum({m} * r.n)                                   AS inner_total,
               sum({m} * coalesce(r.n, 1))                      AS left_total
        FROM {tables[s['L']]['fq']} AS _b LEFT JOIN r ON {on}
    """
    b, mt, it, lt = con.execute(q).fetchone()
    if s["join_type"] == "INNER":
        true_total, joined = mt, it
    else:
        true_total, joined = b, lt
    true_total = true_total or 0
    joined = joined or 0
    return {"true_total": true_total, "joined_total": joined,
            "overstatement": joined - true_total, "ignores_where": True}


# --------------------------------------------------------------------------
# Entry point
# --------------------------------------------------------------------------

def review_query(con: duckdb.DuckDBPyConnection, sql: str) -> list[ReviewWarning]:
    try:
        node = _parse_select(con, sql)
    except _Unsupported as e:
        return [ReviewWarning("UNSUPPORTED", "info", str(e))]
    except Exception as e:  # parser errors, bad SQL
        return [ReviewWarning("PARSE_ERROR", "error", f"Could not parse query: {e}")]

    ctes = {str(k).lower() for k in ((node.get("cte_map") or {}).get("map") or [])
            if isinstance(k, str)}
    ctes |= {str(e.get("key")).lower() for e in ((node.get("cte_map") or {}).get("map") or [])
             if isinstance(e, dict) and e.get("key")}
    tables, joins = {}, []
    try:
        _flatten_from(node.get("from_table"), tables, joins, ctes)
    except _Unsupported as e:
        return [ReviewWarning("UNSUPPORTED", "info", str(e))]

    warnings: list[ReviewWarning] = []
    try:
        cols = {a: _describe(con, t["fq"]) for a, t in tables.items()}
    except duckdb.Error as e:
        return [ReviewWarning("CHECK_FAILED", "error", f"Could not describe tables: {e}")]

    stats = []
    for j in joins:
        try:
            stats.append(_join_stats(con, j, tables, cols, warnings))
        except duckdb.Error as e:
            stats.append(None)
            warnings.append(ReviewWarning("CHECK_FAILED", "warning", f"Join check failed: {e}"))
    for check in (lambda: _check_where(con, node, tables, cols, joins, stats, warnings),
                  lambda: _check_aggregates(con, node, tables, cols, stats, warnings)):
        try:
            check()
        except duckdb.Error as e:
            warnings.append(ReviewWarning("CHECK_FAILED", "warning", f"Check failed: {e}"))
    return warnings

In [24]:
queries_to_review = {
    "Phase 1, customers only (correct, 801)": query_customers,
    "Phase 1, customers JOIN orders (the 1,240)": query_joined,
    "Phase 2, WHERE region != 'West' (drops NULLs)": query_naive,
    "Phase 2, WHERE region IS DISTINCT FROM 'West' (fixed)": query_meant,
    "Phase 2, window function": query_window,
}

for name, sql in queries_to_review.items():
    warnings_found = review_query(con, sql)
    print(f"\n=== {name} ===")
    if not warnings_found:
        print("  no warnings")
    for w in warnings_found:
        print(f"  {w}")


=== Phase 1, customers only (correct, 801) ===
  ReviewWarning(code='NULL_FILTER_DROPS_UNKNOWNS', severity='info', message='WHERE COMPARE_EQUAL on customers.region silently drops 525 of 3,500 customers row(s) where it is NULL (three-valued logic: NULL comparisons are UNKNOWN, and WHERE keeps only TRUE).', details={'alias': 'customers', 'table': 'customers', 'column': 'region', 'ops': ['COMPARE_EQUAL'], 'null_rows': 525, 'total_rows': 3500, 'negative_predicate': False})

=== Phase 1, customers JOIN orders (the 1,240) ===
  ReviewWarning(code='JOIN_ROW_COUNT_CHANGE', severity='warning', message='INNER JOIN o on c.customer_id = o.customer_id changes row count 3,500 -> 23,111 (+19,611): 2864 key value(s) repeat in orders (up to 72 rows per key)', details={'left_rows': 3500, 'right_rows': 23111, 'left_max_mult': 1, 'right_max_mult': 72, 'left_dup_keys': 0, 'right_dup_keys': 2864, 'inner_rows': 23111, 'left_matched': 3500, 'right_matched': 23111, 'join_type': 'INNER', 'L': 'c', 'R': 'o', 'l

**What the reviewer caught and what it missed**

It would have caught the 1,240 dollars before it reached the board. On the Phase 1 joined query it raised a JOIN_ROW_COUNT_CHANGE warning and two AGGREGATE_OVER_FANOUT errors, one for AVG(c.revenue), noting that customer rows were duplicated up to 72 times. On the Phase 2 query WHERE region != 'West' it raised a NULL_FILTER_DROPS_UNKNOWNS warning for the 525 customers with no region. It stayed silent on the corrected IS DISTINCT FROM query and on the window function, so it produced no false alarms on queries that were right.

It also had two weaknesses. First, it treated WHERE region = 'West' as only "info" while giving WHERE region != 'West' a "warning", even though both drop the same 525 customers. That understates a real problem: some of those 525 are West customers, so the "correct" Phase 1 average now leaves part of the West out. Second, it measured the fan-out on the whole tables (3,500 customers to 23,111 rows) rather than on the rows the query actually used (2,200 West customers to 14,655 rows), so the size of the problem it reports is not the query's own.

### AI Transcript
**Prompt (as sent):**
~~~
[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.
~~~

**Reply (unedited):**
~~~
I'll start by checking whether your Phase 1 queries or table data were uploaded, and whether DuckDB is available here.

I can't answer the Evaluate question yet. Your two Phase 1 queries and the $1,240 discrepancy aren't in this conversation: no files were uploaded, and memory is off, so I can't see earlier sessions. I also couldn't run anything, because DuckDB isn't installed in this sandbox and it has no network access. The code passes a syntax check, but neither file has been run. If you paste the two queries and either the real tables or the `CREATE TABLE` statements, I'll trace each one through the checks and tell you which warning, if any, would have fired.

**How `review_query.py` works.** It parses the SELECT with DuckDB's own parser (`json_serialize_sql`), so there's no regex guessing. It returns a list of `ReviewWarning` objects (`code`, `severity`, `message`, `details`). Each warning includes numbers, like rows gained or lost, NULL counts, and how much a fanned-out SUM overstates the total. That means you can test the numbers, not just whether a warning appeared. It also never returns an empty list just because it gave up: bad SQL comes back as `PARSE_ERROR`, and subqueries and CTEs in FROM come back as `UNSUPPORTED`.

**Fan-out is checked without running the join.** An equi-join's row count equals the sum over each key of left count × right count. So the code groups each table by its join key and joins those two small summaries, with one row per distinct key instead of one per joined row. The answer is exact, and the same summary also calculates how much a SUM is inflated.

**Why the NULL filter is a correctness issue.** A comparison with NULL returns UNKNOWN, and WHERE keeps only rows that evaluate to TRUE. So `status <> 'cancelled'` quietly means "known and not cancelled," and the query's real meaning differs from how it reads. The severity depends on intent:
- **info:** a positive filter (`= 'paid'`) dropping NULLs, which is usually intended.
- **warning:** a negative filter (`<>`, `NOT IN`, `NOT LIKE`), which was written to keep everything else.
- **error:** a filter on the right-hand table of a LEFT JOIN, which silently turns it into an INNER JOIN.

The test fixture caught one bug while I was writing it. An INNER join that duplicated one order and dropped a NULL-key order came out at 6 rows → 6 rows, so a totals-only check passed it. The join check now flags duplication and drops separately, even when they cancel out.

**What to verify first when you run it.** I wrote the parse-tree field names (`from_table`, `COMPARE_NOTEQUAL`, `count_star`, and so on) from memory of DuckDB's format, not against a live run. Start with `pytest test_review_query.py`. If a check comes back silent, print `json_serialize_sql` for that query and compare the field names.

**Known limits:**
- The SUM-overstatement figure ignores WHERE filters (`details["ignores_where"]`).
- Chains of several joins are checked one pair at a time.
- CTEs and subqueries in FROM are reported, not analyzed.

If your $1,240 came from a duplicated customer row inflating `SUM(o.amount)`, or from a `<>` filter dropping NULL-status orders, the fixture shows both of those patterns being caught.
~~~

[The reply included the file review_query.py, shown below in its final version.]
~~~
"""
review_query(con, sql) -> list[ReviewWarning]

Data-aware review of a single SELECT against a live DuckDB connection.
Dependencies: duckdb, pandas (plus the standard library).

Checks
  JOIN_ROW_COUNT_CHANGE      a join makes the result grain differ from the driving table
  LEFT_JOIN_NULLIFIED        WHERE filters the nullable side of an outer join (silently becomes INNER)
  NULL_FILTER_DROPS_UNKNOWNS WHERE predicate on a nullable column discards NULL rows
  AGGREGATE_OVER_FANOUT      SUM/AVG/COUNT over rows that a join has duplicated
Meta (so an empty list always means "checked and clean", never "gave up quietly")
  PARSE_ERROR, UNSUPPORTED, CHECK_FAILED

Parsing uses DuckDB's own parser via json_serialize_sql(), so there is no
regex guessing about SQL and no extra dependency like sqlglot.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field

import duckdb
import pandas as pd


@dataclass
class ReviewWarning:
    code: str
    severity: str  # "error" | "warning" | "info"
    message: str
    details: dict = field(default_factory=dict)


def warnings_to_frame(warnings: list[ReviewWarning]) -> pd.DataFrame:
    """Convenience for notebooks / assertions on columns."""
    return pd.DataFrame([asdict(w) for w in warnings],
                        columns=["code", "severity", "message", "details"])


class _Unsupported(Exception):
    pass


# --------------------------------------------------------------------------
# AST helpers
# --------------------------------------------------------------------------

# Aggregates whose result changes when input rows are duplicated.
# MIN/MAX/ANY_VALUE are deliberately absent: duplicating a row can't move an extreme.
_FANOUT_SENSITIVE_AGGS = {
    "sum", "avg", "mean", "count", "count_star", "median", "quantile_cont",
    "string_agg", "list", "array_agg", "stddev", "stddev_samp", "variance",
}

# Predicates that treat NULL as a value rather than as "unknown".
_NULL_SAFE_COMPARES = {"COMPARE_DISTINCT_FROM", "COMPARE_NOT_DISTINCT_FROM"}
_NULL_AWARE_TYPES = {"OPERATOR_IS_NULL", "OPERATOR_IS_NOT_NULL", "OPERATOR_COALESCE"}
_NULL_AWARE_FUNCS = {"coalesce", "ifnull", "isnull", "nvl"}

_NEGATIVE_COMPARES = {"COMPARE_NOTEQUAL", "COMPARE_NOT_IN"}
_PATTERN_FUNCS = {"~~", "!~~", "~~*", "!~~*", "like", "ilike", "not_like",
                  "not_ilike", "regexp_matches", "prefix", "suffix", "contains"}
_NEGATIVE_FUNCS = {"!~~", "!~~*", "not_like", "not_ilike"}


def _q(ident: str) -> str:
    return '"' + str(ident).replace('"', '""') + '"'


def _children(node):
    for v in node.values():
        if isinstance(v, dict) and "class" in v:
            yield v
        elif isinstance(v, list):
            for x in v:
                if isinstance(x, dict) and "class" in x:
                    yield x


def _walk(node):
    # Subqueries hold a "subquery" statement (no "class" key), so they are not
    # descended into. That keeps column resolution in the outer query's scope.
    if not isinstance(node, dict):
        return
    yield node
    for c in _children(node):
        yield from _walk(c)


def _is_colref(n) -> bool:
    return isinstance(n, dict) and n.get("class") == "COLUMN_REF"


def _direct_columns(n):
    """Column refs that are immediate operands (looking through CAST, which preserves NULL)."""
    for c in _children(n):
        while c.get("class") == "CAST" and isinstance(c.get("child"), dict):
            c = c["child"]
        if _is_colref(c):
            yield c


def _parse_select(con, sql: str) -> dict:
    raw = con.execute("SELECT json_serialize_sql(?::VARCHAR)", [sql]).fetchone()[0]
    tree = json.loads(raw)
    if tree.get("error"):
        raise ValueError(tree.get("error_message", "parser error"))
    stmts = tree.get("statements", [])
    if len(stmts) != 1:
        raise _Unsupported(f"expected exactly one statement, got {len(stmts)}")
    node = stmts[0].get("node", {})
    if node.get("type") != "SELECT_NODE":
        raise _Unsupported(f"top-level node is {node.get('type')}; only plain SELECT "
                           "(no UNION/INTERSECT) is reviewed")
    return node


def _flatten_from(ref, tables: dict, joins: list, ctes: set) -> list[str]:
    """Collect alias -> table and the join tree. Returns aliases under this ref."""
    if ref is None:
        return []
    t = ref.get("type")
    if t == "BASE_TABLE":
        name = ref["table_name"]
        if name.lower() in ctes:
            raise _Unsupported(f"'{name}' is a CTE; cardinality checks need base tables")
        alias = (ref.get("alias") or name).lower()
        parts = [p for p in (ref.get("catalog_name"), ref.get("schema_name"), name) if p]
        tables[alias] = {"name": name, "fq": ".".join(_q(p) for p in parts)}
        return [alias]
    if t == "JOIN":
        left = _flatten_from(ref.get("left"), tables, joins, ctes)
        right = _flatten_from(ref.get("right"), tables, joins, ctes)
        joins.append({
            "join_type": (ref.get("join_type") or "INNER").upper(),
            "ref_type": (ref.get("ref_type") or "REGULAR").upper(),
            "condition": ref.get("condition"),
            "using": [u.lower() for u in ref.get("using_columns") or []],
            "left": left,
            "right": right,
        })
        return left + right
    raise _Unsupported(f"FROM item of type {t} (subquery/table function) is not reviewed")


def _describe(con, fq: str) -> dict:
    df = con.execute(f"DESCRIBE {fq}").df()
    return {
        str(r.column_name).lower(): {"type": r.column_type, "nullable": str(r.null).upper() == "YES"}
        for r in df.itertuples()
    }


def _resolve(colref, aliases, cols):
    names = [str(n).lower() for n in colref.get("column_names", [])]
    if not names:
        return None
    if len(names) >= 2:
        a, c = names[-2], names[-1]
        return (a, c) if a in aliases and c in cols.get(a, {}) else None
    owners = [a for a in aliases if names[0] in cols.get(a, {})]
    return (owners[0], names[0]) if len(owners) == 1 else None


def _equi_keys(join, cols):
    """[(left_alias, left_col, right_alias, right_col)], plus whether the ON clause was purely equalities."""
    left, right = set(join["left"]), set(join["right"])
    keys, pure = [], True
    if join["using"]:
        for c in join["using"]:
            la = [a for a in join["left"] if c in cols[a]]
            ra = [a for a in join["right"] if c in cols[a]]
            if la and ra:
                keys.append((la[0], c, ra[0], c))
        return keys, pure

    def visit(n):
        nonlocal pure
        if n is None:
            return
        if n.get("type") == "CONJUNCTION_AND":
            for c in n.get("children", []):
                visit(c)
            return
        if n.get("type") == "COMPARE_EQUAL" and _is_colref(n.get("left")) and _is_colref(n.get("right")):
            a = _resolve(n["left"], left | right, cols)
            b = _resolve(n["right"], left | right, cols)
            if a and b:
                if a[0] in left and b[0] in right:
                    keys.append((a[0], a[1], b[0], b[1]))
                    return
                if b[0] in left and a[0] in right:
                    keys.append((b[0], b[1], a[0], a[1]))
                    return
        pure = False  # non-equi or single-table predicate inside ON

    visit(join["condition"])
    return keys, pure


# --------------------------------------------------------------------------
# Check 1: join row-count change (fan-out or drop)
# --------------------------------------------------------------------------
#
# HOW FAN-OUT IS DETECTED WITHOUT RUNNING THE JOIN (let alone twice):
#
# For an equi-join, the output row count is fully determined by the key
# histograms of the two sides:
#
#     |L JOIN R|  =  SUM over key k of  count_L(k) * count_R(k)
#
# So instead of materialising L JOIN R (rows = up to |L| * max_mult) and then
# counting it, we GROUP BY the key on each side independently, giving one row
# per distinct key, and join those two small histograms. The cost scales with
# distinct keys, not with the joined row count, and the answer is exact, not
# sampled. From the same histogram join we also get:
#   - max multiplicity per side  -> does either side repeat the other's rows?
#   - matched row totals         -> how many rows an INNER join throws away
# NULL keys fall out naturally: they form their own GROUP BY bucket but can
# never satisfy `=` in the histogram join, exactly as in the real join.
#
# The same histogram also prices the damage for SUM: joining a measure column
# to count_R(k) and computing SUM(measure * count_R(k)) reproduces what the
# fanned-out SUM would report, again without building the fanned-out rows.

def _join_stats(con, join, tables, cols, warnings):
    jt = join["join_type"]
    keys, pure = _equi_keys(join, cols)
    label = f"{jt} JOIN {', '.join(join['right'])}"
    if not keys:
        warnings.append(ReviewWarning(
            "JOIN_ROW_COUNT_CHANGE", "warning",
            f"{label} has no column-equality keys (cross/theta join); output row count "
            "is not bounded by either input and was not verified.",
            {"join_type": jt, "right": join["right"]}))
        return None
    la, ra = {k[0] for k in keys}, {k[2] for k in keys}
    if len(la) != 1 or len(ra) != 1:
        warnings.append(ReviewWarning(
            "UNSUPPORTED", "info",
            f"{label}: join keys span several tables on one side; cardinality not checked.",
            {"keys": keys}))
        return None
    L, R = la.pop(), ra.pop()
    lk, rk = [k[1] for k in keys], [k[3] for k in keys]
    n = len(keys)
    lsel = ", ".join(f"{_q(c)} AS k{i}" for i, c in enumerate(lk))
    rsel = ", ".join(f"{_q(c)} AS k{i}" for i, c in enumerate(rk))
    on = " AND ".join(f"l.k{i} = r.k{i}" for i in range(n))
    nn = " AND ".join(f"k{i} IS NOT NULL" for i in range(n))
    q = f"""
        WITH l AS (SELECT {lsel}, count(*) AS n FROM {tables[L]['fq']} GROUP BY ALL),
             r AS (SELECT {rsel}, count(*) AS n FROM {tables[R]['fq']} GROUP BY ALL),
             m AS (SELECT l.n AS ln, r.n AS rn FROM l JOIN r ON {on})
        SELECT
          (SELECT coalesce(sum(n), 0) FROM l)                  AS left_rows,
          (SELECT coalesce(sum(n), 0) FROM r)                  AS right_rows,
          (SELECT coalesce(max(n), 0) FROM l WHERE {nn})       AS left_max_mult,
          (SELECT coalesce(max(n), 0) FROM r WHERE {nn})       AS right_max_mult,
          (SELECT count(*) FROM l WHERE {nn} AND n > 1)        AS left_dup_keys,
          (SELECT count(*) FROM r WHERE {nn} AND n > 1)        AS right_dup_keys,
          (SELECT coalesce(sum(ln * rn), 0) FROM m)            AS inner_rows,
          (SELECT coalesce(sum(ln), 0) FROM m)                 AS left_matched,
          (SELECT coalesce(sum(rn), 0) FROM m)                 AS right_matched
    """
    row = con.execute(q).df().iloc[0]
    s = {k: int(v) for k, v in row.items()}
    s.update(join_type=jt, L=L, R=R, lk=lk, rk=rk,
             left_aliases=join["left"], right_aliases=join["right"])
    unmatched_l = s["left_rows"] - s["left_matched"]
    unmatched_r = s["right_rows"] - s["right_matched"]
    expected = {
        "INNER": s["inner_rows"],
        "LEFT": s["inner_rows"] + unmatched_l,
        "RIGHT": s["inner_rows"] + unmatched_r,
        "OUTER": s["inner_rows"] + unmatched_l + unmatched_r,
        "FULL": s["inner_rows"] + unmatched_l + unmatched_r,
    }.get(jt, s["inner_rows"])
    baseline = s["right_rows"] if jt == "RIGHT" else s["left_rows"]
    s.update(unmatched_left=unmatched_l, unmatched_right=unmatched_r,
             expected_rows=expected, baseline_rows=baseline)

    # Compare components, not just totals: +1 duplicated row and -1 dropped row
    # nets to the same count but is two separate errors in every aggregate.
    dropped = unmatched_l if jt in ("INNER", "RIGHT") else 0
    grows = s["right_dup_keys"] > 0 or (s["left_dup_keys"] > 0 and jt in ("RIGHT", "FULL", "OUTER"))
    if expected != baseline or grows or dropped:
        reasons = []
        if s["right_dup_keys"]:
            reasons.append(f"{s['right_dup_keys']} key value(s) repeat in {tables[R]['name']} "
                           f"(up to {s['right_max_mult']} rows per key)")
        if s["left_dup_keys"] and jt in ("RIGHT", "FULL", "OUTER"):
            reasons.append(f"{s['left_dup_keys']} key value(s) repeat in {tables[L]['name']}")
        if jt in ("INNER", "RIGHT") and unmatched_l:
            reasons.append(f"{unmatched_l} {tables[L]['name']} row(s) have no match "
                           "(including NULL keys) and are dropped")
        warnings.append(ReviewWarning(
            "JOIN_ROW_COUNT_CHANGE", "warning",
            f"{label} on {', '.join(f'{L}.{a} = {R}.{b}' for a, b in zip(lk, rk))} changes row "
            f"count {baseline:,} -> {expected:,} ({expected - baseline:+,}): " + "; ".join(reasons)
            + ("" if pure else " [ON clause has extra non-equality terms; count is an upper bound]"),
            {k: v for k, v in s.items() if k not in ("left_aliases", "right_aliases")}))
    return s


# --------------------------------------------------------------------------
# Check 2: filters on nullable columns
# --------------------------------------------------------------------------
#
# WHY THIS IS A CORRECTNESS ISSUE, NOT STYLE:
#
# SQL predicates are three-valued: TRUE, FALSE, UNKNOWN. Any comparison with
# NULL (=, <>, <, IN, NOT IN, LIKE...) yields UNKNOWN, and WHERE keeps only
# TRUE rows. So `WHERE status <> 'cancelled'` does NOT mean "everything except
# cancelled"; it means "rows whose status is known and is not cancelled".
# Rows with an unknown status vanish, no error is raised, and the total just
# comes out lower. The query's meaning differs from its reading, so the
# number on the dashboard is wrong, not merely written in an unidiomatic way.
# Worse, the gap is data-dependent: the query is right today with zero NULLs
# and wrong after tomorrow's load introduces some.
#
# Severity follows intent. A positive predicate (status = 'paid') excluding
# NULLs is almost always what the author meant, so it is "info". A negative
# predicate (<>, NOT IN, NOT LIKE) is written to *keep* everything else, so
# losing NULLs contradicts its intent and is a "warning". A predicate on the
# nullable side of a LEFT JOIN is an "error": it turns the LEFT JOIN into an
# INNER JOIN, dropping every unmatched row regardless of the column's own data.
#
# Columns already wrapped in IS [NOT] NULL / COALESCE in the WHERE are treated
# as deliberately handled.

def _check_where(con, node, tables, cols, joins, stats, warnings):
    where = node.get("where_clause")
    if not where:
        return
    aliases = list(tables)
    handled, filtered = set(), {}

    def visit(n, negated):
        if not isinstance(n, dict):
            return
        t = n.get("type", "") or ""
        fn = (n.get("function_name") or "").lower()
        if t in _NULL_AWARE_TYPES or fn in _NULL_AWARE_FUNCS:
            for c in _walk(n):
                if _is_colref(c) and (r := _resolve(c, aliases, cols)):
                    handled.add(r)
            return
        if t == "OPERATOR_NOT":
            for c in _children(n):
                visit(c, not negated)
            return
        is_cmp = t.startswith("COMPARE_") and t not in _NULL_SAFE_COMPARES
        if is_cmp or fn in _PATTERN_FUNCS:
            neg = (t in _NEGATIVE_COMPARES or fn in _NEGATIVE_FUNCS) != negated
            for c in _direct_columns(n):
                if r := _resolve(c, aliases, cols):
                    filtered.setdefault(r, {"negative": False, "ops": set()})
                    filtered[r]["negative"] |= neg
                    filtered[r]["ops"].add(t or fn)
        for c in _children(n):
            visit(c, negated)

    visit(where, False)

    nullable_side = {}
    for j in joins:
        if j["join_type"] in ("LEFT", "FULL", "OUTER"):
            for a in j["right"]:
                nullable_side[a] = j
        if j["join_type"] in ("RIGHT", "FULL", "OUTER"):
            for a in j["left"]:
                nullable_side[a] = j

    for (alias, col), info in filtered.items():
        if (alias, col) in handled:
            continue
        tname, ops = tables[alias]["name"], sorted(info["ops"])
        if alias in nullable_side:
            j = nullable_side[alias]
            s = next((s for s in stats if s and alias in (s["R"], s["L"])), None)
            lost = s["unmatched_left"] if s and alias == s["R"] else (s["unmatched_right"] if s else None)
            warnings.append(ReviewWarning(
                "LEFT_JOIN_NULLIFIED", "error",
                f"WHERE filters {alias}.{col}, which is on the NULL-extended side of a "
                f"{j['join_type']} JOIN. Unmatched rows have {alias}.{col} = NULL, the predicate is "
                "UNKNOWN, and they are dropped: the outer join is effectively INNER"
                + (f" ({lost:,} row(s) lost)." if lost is not None else ".")
                + " Move the predicate into the ON clause or handle NULL explicitly.",
                {"alias": alias, "table": tname, "column": col, "ops": ops, "rows_lost": lost}))
            continue
        meta = cols[alias][col]
        nulls, total = con.execute(
            f"SELECT count(*) FILTER (WHERE {_q(col)} IS NULL), count(*) FROM {tables[alias]['fq']}"
        ).fetchone()
        nulls, total = int(nulls), int(total)
        if nulls == 0 and not meta["nullable"]:
            continue
        if nulls == 0:
            sev, msg = "info", (f"{alias}.{col} is declared nullable and has no NULLs today; "
                                f"the {', '.join(ops)} predicate will start dropping rows when it does.")
        else:
            sev = "warning" if info["negative"] else "info"
            msg = (f"WHERE {', '.join(ops)} on {alias}.{col} silently drops {nulls:,} of {total:,} "
                   f"{tname} row(s) where it is NULL (three-valued logic: NULL comparisons are UNKNOWN, "
                   "and WHERE keeps only TRUE)."
                   + (" The predicate is negative, so the author almost certainly meant to keep them."
                      if info["negative"] else ""))
        warnings.append(ReviewWarning("NULL_FILTER_DROPS_UNKNOWNS", sev, msg,
                                      {"alias": alias, "table": tname, "column": col, "ops": ops,
                                       "null_rows": nulls, "total_rows": total,
                                       "negative_predicate": info["negative"]}))


# --------------------------------------------------------------------------
# Check 3: aggregates over fanned-out rows
# --------------------------------------------------------------------------

def _check_aggregates(con, node, tables, cols, stats, warnings):
    # Which aliases have their rows repeated by some join?
    fanned = {}
    for s in stats:
        if not s:
            continue
        if s["right_max_mult"] > 1 and s["join_type"] in ("INNER", "LEFT", "FULL", "OUTER"):
            for a in s["left_aliases"]:
                fanned.setdefault(a, s)
        if s["left_max_mult"] > 1 and s["join_type"] in ("INNER", "RIGHT", "FULL", "OUTER"):
            for a in s["right_aliases"]:
                fanned.setdefault(a, s)
    if not fanned:
        return

    exprs = list(node.get("select_list") or [])
    if node.get("having"):
        exprs.append(node["having"])
    aliases = list(tables)
    for e in exprs:
        for f in _walk(e):
            fn = (f.get("function_name") or "").lower()
            if f.get("class") != "FUNCTION" or fn not in _FANOUT_SENSITIVE_AGGS or f.get("distinct"):
                continue
            if fn == "count_star":
                s = next(iter(fanned.values()))
                warnings.append(ReviewWarning(
                    "AGGREGATE_OVER_FANOUT", "error",
                    f"COUNT(*) counts joined rows, and {tables[s['R']]['name']} repeats "
                    f"{tables[s['L']]['name']} rows; it is not a count of {tables[s['L']]['name']}.",
                    {"function": fn, "join": f"{s['L']} -> {s['R']}"}))
                continue
            for c in _walk(f):
                r = _resolve(c, aliases, cols) if _is_colref(c) else None
                if not r or r[0] not in fanned:
                    continue
                s = fanned[r[0]]
                details = {"function": fn, "alias": r[0], "column": r[1],
                           "join": f"{s['L']} -> {s['R']}", "max_multiplicity":
                           max(s["right_max_mult"], s["left_max_mult"])}
                msg = (f"{fn.upper()}({r[0]}.{r[1]}) runs after a join that duplicates "
                       f"{tables[r[0]]['name']} rows (up to {details['max_multiplicity']}x)")
                arg = (f.get("children") or [None])[0]
                while isinstance(arg, dict) and arg.get("class") == "CAST":
                    arg = arg.get("child")
                if fn == "sum" and _is_colref(arg) and r[0] == s["L"] and s["right_max_mult"] > 1:
                    d = _price_sum_inflation(con, tables, s, r[1])
                    details.update(d)
                    msg += (f"; the joined SUM overstates by {d['overstatement']:,} "
                            f"({d['true_total']:,} -> {d['joined_total']:,}, before WHERE filters)")
                warnings.append(ReviewWarning("AGGREGATE_OVER_FANOUT", "error", msg + ".", details))


def _price_sum_inflation(con, tables, s, col):
    # Same histogram trick: multiply each measure by how many partner rows it
    # will be paired with, instead of materialising the pairs.
    n = len(s["lk"])
    rsel = ", ".join(f"{_q(c)} AS k{i}" for i, c in enumerate(s["rk"]))
    on = " AND ".join(f"_b.{_q(c)} = r.k{i}" for i, c in enumerate(s["lk"]))
    m = f"_b.{_q(col)}"
    q = f"""
        WITH r AS (SELECT {rsel}, count(*) AS n FROM {tables[s['R']]['fq']} GROUP BY ALL)
        SELECT sum({m})                                         AS base_total,
               sum(CASE WHEN r.n IS NOT NULL THEN {m} END)      AS matched_total,
               sum({m} * r.n)                                   AS inner_total,
               sum({m} * coalesce(r.n, 1))                      AS left_total
        FROM {tables[s['L']]['fq']} AS _b LEFT JOIN r ON {on}
    """
    b, mt, it, lt = con.execute(q).fetchone()
    if s["join_type"] == "INNER":
        true_total, joined = mt, it
    else:
        true_total, joined = b, lt
    true_total = true_total or 0
    joined = joined or 0
    return {"true_total": true_total, "joined_total": joined,
            "overstatement": joined - true_total, "ignores_where": True}


# --------------------------------------------------------------------------
# Entry point
# --------------------------------------------------------------------------

def review_query(con: duckdb.DuckDBPyConnection, sql: str) -> list[ReviewWarning]:
    try:
        node = _parse_select(con, sql)
    except _Unsupported as e:
        return [ReviewWarning("UNSUPPORTED", "info", str(e))]
    except Exception as e:  # parser errors, bad SQL
        return [ReviewWarning("PARSE_ERROR", "error", f"Could not parse query: {e}")]

    ctes = {str(k).lower() for k in ((node.get("cte_map") or {}).get("map") or [])
            if isinstance(k, str)}
    ctes |= {str(e.get("key")).lower() for e in ((node.get("cte_map") or {}).get("map") or [])
             if isinstance(e, dict) and e.get("key")}
    tables, joins = {}, []
    try:
        _flatten_from(node.get("from_table"), tables, joins, ctes)
    except _Unsupported as e:
        return [ReviewWarning("UNSUPPORTED", "info", str(e))]

    warnings: list[ReviewWarning] = []
    try:
        cols = {a: _describe(con, t["fq"]) for a, t in tables.items()}
    except duckdb.Error as e:
        return [ReviewWarning("CHECK_FAILED", "error", f"Could not describe tables: {e}")]

    stats = []
    for j in joins:
        try:
            stats.append(_join_stats(con, j, tables, cols, warnings))
        except duckdb.Error as e:
            stats.append(None)
            warnings.append(ReviewWarning("CHECK_FAILED", "warning", f"Join check failed: {e}"))
    for check in (lambda: _check_where(con, node, tables, cols, joins, stats, warnings),
                  lambda: _check_aggregates(con, node, tables, cols, stats, warnings)):
        try:
            check()
        except duckdb.Error as e:
            warnings.append(ReviewWarning("CHECK_FAILED", "warning", f"Check failed: {e}"))
    return warnings
~~~

### Task 4.2: The one-page writeup

**To:** Director
**Re:** West-region revenue per customer, and the account-management programme

**Which number is right.** Finance's figure is right. In my reproduction, average revenue per West customer is about 801 dollars, averaged over 2,200 customers, one row each.

**Why the two differed.** The dashboard query joined customers to orders before averaging. That turns each customer into one row per order, so a customer with 30 orders counts 30 times. Because customers who order more also spend more, the average rose to about 1,214 dollars, a number about heavy buyers, not customers. No arithmetic was wrong; the query silently changed what it was averaging over. A second, quieter problem: 15% of customers have no recorded region, and region filters drop them without warning.

**Whether the programme works.** We cannot tell from the West's higher revenue. West customers had larger accounts to begin with. In a simulation where the true effect is 50 dollars, the simple comparison says 335; adjusting for account size gives 57. Controlling for support tickets, which the programme itself increases, flips the answer to minus 55. Any real estimate needs data on account size before the programme started, and ideally a randomised rollout.

**What I recommend.** Before a number goes on a slide: (1) check that every join keeps one row per unit being averaged; (2) report how many rows have missing values in any filtered column; (3) control only for characteristics fixed before the programme began.